<a href="https://colab.research.google.com/github/Sushrut0202/CIIMS/blob/main/Automated_NGS_sample_selection_env.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

INPUT_FILE = "NEERI_updated.xlsx"
OUTPUT_FILE = "Environmental_mNGS_selection.xlsx"

# -----------------------------
# LOAD WITH CORRECT HEADER
# -----------------------------

raw = pd.read_excel(INPUT_FILE, header=None)

# first row = main header
# second row = antibiotic names

header1 = raw.iloc[0]
header2 = raw.iloc[1]

new_cols = []

for h1, h2 in zip(header1, header2):

    if pd.notna(h2):
        new_cols.append(str(h2))
    else:
        new_cols.append(str(h1))

df = raw.iloc[2:].copy()
df.columns = new_cols

print(f"Shape after loading and renaming columns: {df.shape}")

# -----------------------------
# CLEAN BASIC FIELDS
# -----------------------------

df["Pathogen"] = (
    df["Pathogen"] # Changed from "ESKAPE Pathogen"
    .astype(str)
    .str.lower()
    .str.strip()
)

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "a. baumannii",
    "p. aeruginosa"
]

df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

print(f"Shape after filtering target pathogens: {df.shape}")

# -----------------------------
# COUNT RESISTANCE
# -----------------------------

metadata_cols = [
    "Sr. No",
    "Sample Code",
    "Farm Name",
    "sample type ", # Added trailing space
    "Pathogen" # Changed from "ESKAPE Pathogen"
]

amr_cols = [c for c in df.columns if c not in metadata_cols]

def count_r(row):
    return sum(str(row[c]).strip().upper() == "R" for c in amr_cols)

df["No_of_ABX_Classes_Resistant"] = df.apply(count_r, axis=1)

df = df[df["No_of_ABX_Classes_Resistant"] >= 2]

print(f"Shape after filtering for resistance >= 2: {df.shape}")

# -----------------------------
# REMOVE DUPLICATE SAMPLE PATHOGENS
# -----------------------------

df = (
    df.sort_values("No_of_ABX_Classes_Resistant", ascending=False)
      .drop_duplicates(subset=["Sample Code"], keep="first")
)

print(f"Shape after removing duplicates: {df.shape}")

# -----------------------------
# PI PATHOGEN QUOTA
# -----------------------------

QUOTA = {
    "e. coli":4,
    "k. pneumoniae":3,
    "s. aureus":3,
    "a. baumannii":2,
    "p. aeruginosa":2
}

selected = []

for p, n in QUOTA.items():

    sub = df[df["Pathogen"]==p]

    sub = sub.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    selected.append(sub.head(n))

result = pd.concat(selected).drop_duplicates("Sample Code")

# fill remaining slots if pathogens missing

remaining = 14 - len(result)

if remaining > 0:

    leftover = df[~df["Sample Code"].isin(result["Sample Code"])]

    leftover = leftover.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    result = pd.concat([result, leftover.head(remaining)])

print(f"Final result shape: {result.shape}")

# -----------------------------
# FINAL OUTPUT
# -----------------------------

final_cols = [
    "Sample Code",
    "Farm Name",
    "sample type ", # Added trailing space
    "Pathogen",
    "No_of_ABX_Classes_Resistant"
]

result = result[final_cols]

result.to_excel(OUTPUT_FILE, index=False)

print("Environmental selection complete")

Shape after loading and renaming columns: (313, 57)
Shape after filtering target pathogens: (170, 57)
Shape after filtering for resistance >= 2: (0, 58)
Shape after removing duplicates: (0, 58)
Final result shape: (0, 58)
Environmental selection complete


In [ ]:
display(result)

,Sample Code,Farm Name,sample type,Pathogen,No_of_ABX_Classes_Resistant


In [ ]:
import pandas as pd

INPUT_FILE = "NEERI_updated.xlsx"
OUTPUT_FILE = "Environmental_mNGS_selection.xlsx"

# ---------------------------------------------------
# 1. LOAD FILE AND FIX HEADERS
# ---------------------------------------------------

raw = pd.read_excel(INPUT_FILE, header=None)

header1 = raw.iloc[0]
header2 = raw.iloc[1]

cols = []

for h1, h2 in zip(header1, header2):

    if pd.notna(h2) and "Unnamed" not in str(h2):
        cols.append(str(h2).strip())
    else:
        cols.append(str(h1).strip())

df = raw.iloc[2:].copy()
df.columns = cols

# remove empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# clean column names
df.columns = df.columns.str.strip()

print("Shape after loading:", df.shape)

# ---------------------------------------------------
# 2. NORMALIZE PATHOGEN NAMES
# ---------------------------------------------------

df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(".", "", regex=False)
)

df["Pathogen"] = df["Pathogen"].replace({
    "e coli": "e. coli",
    "k pneumoniae": "k. pneumoniae",
    "s aureus": "s. aureus",
    "p aeruginosa": "p. aeruginosa",
    "a baumannii": "a. baumannii"
})

print("Unique pathogens found:")
print(df["Pathogen"].unique())

# ---------------------------------------------------
# 3. KEEP ONLY TARGET PATHOGENS
# ---------------------------------------------------

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "a. baumannii",
    "p. aeruginosa"
]

df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

print("After pathogen filtering:", df.shape)

# ---------------------------------------------------
# 4. DETECT ANTIBIOTIC COLUMNS
# ---------------------------------------------------

metadata_cols = [
    "Sr. No",
    "Sample Code",
    "Farm Name",
    "Sample type [Lactating Cattle]",
    "Pathogen"
]

amr_cols = [c for c in df.columns if c not in metadata_cols]

print("AMR columns detected:", len(amr_cols))

# ---------------------------------------------------
# 5. COUNT RESISTANCE
# ---------------------------------------------------

def count_r(row):
    return sum(str(row[c]).strip().upper() == "R" for c in amr_cols)

df["No_of_ABX_Classes_Resistant"] = df.apply(count_r, axis=1)

df = df[df["No_of_ABX_Classes_Resistant"] >= 2]

print("After resistance filtering:", df.shape)

# ---------------------------------------------------
# 6. REMOVE DUPLICATE SAMPLE CODES
# ---------------------------------------------------

df = (
    df.sort_values("No_of_ABX_Classes_Resistant", ascending=False)
      .drop_duplicates(subset=["Sample Code"], keep="first")
)

print("After duplicate removal:", df.shape)

# ---------------------------------------------------
# 7. APPLY PI PATHOGEN QUOTA
# ---------------------------------------------------

QUOTA = {
    "e. coli":4,
    "k. pneumoniae":3,
    "s. aureus":3,
    "a. baumannii":2,
    "p. aeruginosa":2
}

selected = []

for pathogen, n in QUOTA.items():

    subset = df[df["Pathogen"] == pathogen]

    subset = subset.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    selected.append(subset.head(n))

result = pd.concat(selected)

# ---------------------------------------------------
# 8. FILL REMAINING SLOTS (IF SOME PATHOGENS MISSING)
# ---------------------------------------------------

remaining = 14 - len(result)

if remaining > 0:

    leftover = df[~df["Sample Code"].isin(result["Sample Code"])]

    leftover = leftover.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    result = pd.concat([result, leftover.head(remaining)])

# ---------------------------------------------------
# 9. FINAL CLEAN OUTPUT
# ---------------------------------------------------

result = result[[
    "Sample Code",
    "Farm Name",
    "Sample type [Lactating Cattle]",
    "Pathogen",
    "No_of_ABX_Classes_Resistant"
]]

result = result.sort_values(
    "No_of_ABX_Classes_Resistant",
    ascending=False
)

result.to_excel(OUTPUT_FILE, index=False)

print("\nEnvironmental selection complete")
print("Final dataset size:", result.shape)


Shape after loading: (313, 57)
Unique pathogens found:
['nan' 'e. coli' 's. aureus' 'k. pneumoniae' 's pneumoniae' 'abaumannii'
 'p. aeruginosa' 'ecoli' 'alwfii' 'calbicans' 's epidermidis']
After pathogen filtering: (170, 57)
AMR columns detected: 53
After resistance filtering: (0, 58)
After duplicate removal: (0, 58)


KeyError: "['Sample type [Lactating Cattle]'] not in index"

In [ ]:
import pandas as pd

INPUT_FILE = "NEERI_updated.xlsx"
OUTPUT_FILE = "Environmental_mNGS_selection.xlsx"

# ---------------------------------------------------
# 1. LOAD FILE AND FIX HEADERS
# ---------------------------------------------------

raw = pd.read_excel(INPUT_FILE, header=None)

header1 = raw.iloc[0]
header2 = raw.iloc[1]

cols = []

for h1, h2 in zip(header1, header2):

    if pd.notna(h2) and "Unnamed" not in str(h2):
        cols.append(str(h2).strip())
    else:
        cols.append(str(h1).strip())

df = raw.iloc[2:].copy()
df.columns = cols

# remove empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# clean column names
df.columns = df.columns.str.strip()

print("Shape after loading:", df.shape)

# ---------------------------------------------------
# 2. NORMALIZE PATHOGEN NAMES
# ---------------------------------------------------

df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(".", "", regex=False)
)

df["Pathogen"] = df["Pathogen"].replace({
    "e coli": "e. coli",
    "k pneumoniae": "k. pneumoniae",
    "s aureus": "s. aureus",
    "p aeruginosa": "p. aeruginosa",
    "a baumannii": "a. baumannii"
})

print("Unique pathogens found:")
print(df["Pathogen"].unique())

# ---------------------------------------------------
# 3. KEEP ONLY TARGET PATHOGENS
# ---------------------------------------------------

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "a. baumannii",
    "p. aeruginosa"
]

df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

print("After pathogen filtering:", df.shape)


# ---------------------------------------------------
# 4. DETECT ANTIBIOTIC COLUMNS
# ---------------------------------------------------

start_idx = df.columns.get_loc("List of Antimicrograms")
amr_cols = df.columns[start_idx+1:]

print("Detected antibiotic columns:", len(amr_cols))

# ---------------------------------------------------
# 5. COUNT RESISTANCE PER ISOLATE
# ---------------------------------------------------

def count_r(row):
    return sum(str(row[c]).strip().upper() == "R" for c in amr_cols)

df["No_of_ABX_Classes_Resistant"] = df.apply(count_r, axis=1)

df = df[df["No_of_ABX_Classes_Resistant"] >= 2]

print("After resistance filtering:", df.shape)



# ---------------------------------------------------
# 6. REMOVE DUPLICATE SAMPLE CODES
# ---------------------------------------------------

df = (
    df.sort_values("No_of_ABX_Classes_Resistant", ascending=False)
      .drop_duplicates(subset=["Isolate code"], keep="first")

)

print("After duplicate removal:", df.shape)

# ---------------------------------------------------
# 7. APPLY PI PATHOGEN QUOTA
# ---------------------------------------------------

QUOTA = {
    "e. coli":4,
    "k. pneumoniae":3,
    "s. aureus":3,
    "a. baumannii":2,
    "p. aeruginosa":2
}

selected = []

for pathogen, n in QUOTA.items():

    subset = df[df["Pathogen"] == pathogen]

    subset = subset.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    selected.append(subset.head(n))

result = pd.concat(selected)

# ---------------------------------------------------
# 8. FILL REMAINING SLOTS (IF SOME PATHOGENS MISSING)
# ---------------------------------------------------

remaining = 14 - len(result)

if remaining > 0:

    leftover = df[~df["Sample Code"].isin(result["Sample Code"])]

    leftover = leftover.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    result = pd.concat([result, leftover.head(remaining)])

# ---------------------------------------------------
# 9. FINAL CLEAN OUTPUT
# ---------------------------------------------------

result = result[[
    "Sample Code",
    "Farm Name",
    "Sample type [Lactating Cattle]",
    "Pathogen",
    "No_of_ABX_Classes_Resistant"
]]

result = result.sort_values(
    "No_of_ABX_Classes_Resistant",
    ascending=False
)

result.to_excel(OUTPUT_FILE, index=False)

print("\nEnvironmental selection complete")
print("Final dataset size:", result.shape)


Shape after loading: (313, 57)
Unique pathogens found:
['nan' 'e. coli' 's. aureus' 'k. pneumoniae' 's pneumoniae' 'abaumannii'
 'p. aeruginosa' 'ecoli' 'alwfii' 'calbicans' 's epidermidis']
After pathogen filtering: (170, 57)
Detected antibiotic columns: 44
After resistance filtering: (0, 58)
After duplicate removal: (0, 58)


KeyError: "['Sample type [Lactating Cattle]'] not in index"

In [ ]:
import pandas as pd

INPUT_FILE = "NEERI_updated.xlsx"
OUTPUT_FILE = "Environmental_mNGS_selection.xlsx"

# ---------------------------------------------------
# 1. LOAD FILE AND FIX HEADERS
# ---------------------------------------------------

raw = pd.read_excel(INPUT_FILE, header=None)

header1 = raw.iloc[0]
header2 = raw.iloc[1]

cols = []

for h1, h2 in zip(header1, header2):

    if pd.notna(h2) and "Unnamed" not in str(h2):
        cols.append(str(h2).strip())
    else:
        cols.append(str(h1).strip())

df = raw.iloc[2:].copy()
df.columns = cols

# remove empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# clean column names
df.columns = df.columns.str.strip()

print("Shape after loading:", df.shape)


# ---------------------------------------------------
# 2. NORMALIZE PATHOGEN NAMES
# ---------------------------------------------------

df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(".", "", regex=False)
)

df["Pathogen"] = df["Pathogen"].replace({
    "ecoli": "e. coli",
    "e coli": "e. coli",
    "k pneumoniae": "k. pneumoniae",
    "s aureus": "s. aureus",
    "p aeruginosa": "p. aeruginosa",
    "abaumannii": "a. baumannii"
})

print("Unique pathogens found:")
print(df["Pathogen"].unique())


# ---------------------------------------------------
# 3. KEEP ONLY TARGET PATHOGENS
# ---------------------------------------------------

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "a. baumannii",
    "p. aeruginosa"
]

df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

print("After pathogen filtering:", df.shape)


# ---------------------------------------------------
# 4. DETECT ANTIBIOTIC COLUMNS
# ---------------------------------------------------

start_idx = df.columns.get_loc("List of Antimicrograms")

amr_cols = df.columns[start_idx+1:]

print("Detected antibiotic columns:", len(amr_cols))


# ---------------------------------------------------
# 5. COUNT RESISTANCE PER ISOLATE
# ---------------------------------------------------

def count_r(row):
    return sum(
        str(row[c]).strip().upper().startswith("R")
        for c in amr_cols
    )

df["No_of_ABX_Classes_Resistant"] = df.apply(count_r, axis=1)

df = df[df["No_of_ABX_Classes_Resistant"] >= 2]

print("After resistance filtering:", df.shape)


# ---------------------------------------------------
# 6. REMOVE DUPLICATE ISOLATES
# ---------------------------------------------------

df = (
    df.sort_values("No_of_ABX_Classes_Resistant", ascending=False)
      .drop_duplicates(subset=["Isolate code"], keep="first")
)

print("After duplicate removal:", df.shape)


# ---------------------------------------------------
# 7. APPLY PI PATHOGEN QUOTA
# ---------------------------------------------------

QUOTA = {
    "e. coli":4,
    "k. pneumoniae":3,
    "s. aureus":3,
    "a. baumannii":2,
    "p. aeruginosa":2
}

selected = []

for pathogen, n in QUOTA.items():

    subset = df[df["Pathogen"] == pathogen]

    subset = subset.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    selected.append(subset.head(n))

result = pd.concat(selected)


# ---------------------------------------------------
# 8. FILL REMAINING SLOTS
# ---------------------------------------------------

remaining = 14 - len(result)

if remaining > 0:

    leftover = df[~df["Isolate code"].isin(result["Isolate code"])]

    leftover = leftover.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    result = pd.concat([result, leftover.head(remaining)])


# ---------------------------------------------------
# 9. FINAL CLEAN OUTPUT
# ---------------------------------------------------

# detect sample type column automatically
sample_type_col = [c for c in df.columns if "sample type" in c.lower()][0]

result = result[[
    "Sample Code",
    "Isolate code",
    "Farm Name",
    sample_type_col,
    "Pathogen",
    "No_of_ABX_Classes_Resistant"
]]

result = result.sort_values(
    "No_of_ABX_Classes_Resistant",
    ascending=False
)

result.to_excel(OUTPUT_FILE, index=False)

print("\nEnvironmental selection complete")
print("Final dataset size:", result.shape)


Shape after loading: (313, 57)
Unique pathogens found:
['nan' 'e. coli' 's. aureus' 'k. pneumoniae' 's pneumoniae' 'a. baumannii'
 'p. aeruginosa' 'alwfii' 'calbicans' 's epidermidis']
After pathogen filtering: (253, 57)
Detected antibiotic columns: 44
After resistance filtering: (0, 58)
After duplicate removal: (0, 58)

Environmental selection complete
Final dataset size: (0, 6)


In [ ]:
import pandas as pd

INPUT_FILE = "NEERI_updated.xlsx"
OUTPUT_FILE = "Environmental_mNGS_selection.xlsx"

# ---------------------------------------------------
# 1. LOAD FILE AND FIX HEADERS
# ---------------------------------------------------

raw = pd.read_excel(INPUT_FILE, header=None)

header1 = raw.iloc[0]
header2 = raw.iloc[1]

cols = []

for h1, h2 in zip(header1, header2):

    if pd.notna(h2) and "Unnamed" not in str(h2):
        cols.append(str(h2).strip())
    else:
        cols.append(str(h1).strip())

df = raw.iloc[2:].copy()
df.columns = cols

# remove empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# clean column names
df.columns = df.columns.str.strip()

print("Shape after loading:", df.shape)


# ---------------------------------------------------
# 2. NORMALIZE PATHOGEN NAMES
# ---------------------------------------------------

df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(".", "", regex=False)
)

df["Pathogen"] = df["Pathogen"].replace({
    "ecoli": "e. coli",
    "e coli": "e. coli",
    "k pneumoniae": "k. pneumoniae",
    "s aureus": "s. aureus",
    "p aeruginosa": "p. aeruginosa",
    "abaumannii": "a. baumannii"
})

print("Unique pathogens found:")
print(df["Pathogen"].unique())


# ---------------------------------------------------
# 3. KEEP ONLY TARGET PATHOGENS
# ---------------------------------------------------

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "a. baumannii",
    "p. aeruginosa"
]

df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

print("After pathogen filtering:", df.shape)


# ---------------------------------------------------
# 4. EXTRACT RESISTANCE FROM AST COLUMN
# ---------------------------------------------------

def extract_resistance(val):

    if pd.isna(val):
        return 0

    val = str(val).lower()

    if "resistant" in val:

        num = val.split()[0]

        try:
            return int(num)
        except:
            return 0

    return 0


df["No_of_ABX_Classes_Resistant"] = df["RESISTANT [AST Profiling]"].apply(extract_resistance)

print("Resistance summary:")
print(df["No_of_ABX_Classes_Resistant"].describe())


# keep isolates with resistance ≥2
df = df[df["No_of_ABX_Classes_Resistant"] >= 2]

print("After resistance filtering:", df.shape)


# ---------------------------------------------------
# 5. REMOVE DUPLICATE ISOLATES
# ---------------------------------------------------

df = (
    df.sort_values("No_of_ABX_Classes_Resistant", ascending=False)
      .drop_duplicates(subset=["Isolate code","Pathogen"], keep="first")
)

print("After duplicate removal:", df.shape)


# ---------------------------------------------------
# 6. APPLY PI PATHOGEN QUOTA
# ---------------------------------------------------

QUOTA = {
    "e. coli":4,
    "k. pneumoniae":3,
    "s. aureus":3,
    "a. baumannii":2,
    "p. aeruginosa":2
}

selected = []

for pathogen, n in QUOTA.items():

    subset = df[df["Pathogen"] == pathogen]

    subset = subset.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    selected.append(subset.head(n))

result = pd.concat(selected)


# ---------------------------------------------------
# 7. FILL REMAINING SLOTS IF NEEDED
# ---------------------------------------------------

remaining = 14 - len(result)

if remaining > 0:

    leftover = df[~df["Isolate code"].isin(result["Isolate code"])]

    leftover = leftover.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    result = pd.concat([result, leftover.head(remaining)])


# ---------------------------------------------------
# 8. FINAL OUTPUT
# ---------------------------------------------------

# detect sample type column automatically
sample_type_col = [c for c in df.columns if "sample type" in c.lower()][0]

result = result[[
    "Sample Code",
    "Isolate code",
    "Farm Name",
    sample_type_col,
    "Pathogen",
    "No_of_ABX_Classes_Resistant"
]]

result = result.sort_values(
    "No_of_ABX_Classes_Resistant",
    ascending=False
)

result.to_excel(OUTPUT_FILE, index=False)

print("\nEnvironmental selection complete")
print("Final dataset size:", result.shape)


Shape after loading: (313, 57)
Unique pathogens found:
['nan' 'e. coli' 's. aureus' 'k. pneumoniae' 's pneumoniae' 'a. baumannii'
 'p. aeruginosa' 'alwfii' 'calbicans' 's epidermidis']
After pathogen filtering: (253, 57)


KeyError: 'RESISTANT [AST Profiling]'

In [ ]:
import pandas as pd

INPUT_FILE = "NEERI_updated.xlsx"
OUTPUT_FILE = "Environmental_mNGS_selection.xlsx"

# ---------------------------------------------------
# 1. LOAD FILE AND FIX HEADERS
# ---------------------------------------------------

raw = pd.read_excel(INPUT_FILE, header=None)

header1 = raw.iloc[0]
header2 = raw.iloc[1]

cols = []

for h1, h2 in zip(header1, header2):

    if pd.notna(h2) and "Unnamed" not in str(h2):
        cols.append(str(h2).strip())
    else:
        cols.append(str(h1).strip())

df = raw.iloc[2:].copy()
df.columns = cols

# remove empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# clean column names
df.columns = df.columns.str.strip()

print("Shape after loading:", df.shape)


# ---------------------------------------------------
# 2. NORMALIZE PATHOGEN NAMES
# ---------------------------------------------------

df["Pathogen"] = (
    df["Pathogen"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(".", "", regex=False)
)

df["Pathogen"] = df["Pathogen"].replace({
    "ecoli": "e. coli",
    "e coli": "e. coli",
    "k pneumoniae": "k. pneumoniae",
    "s aureus": "s. aureus",
    "p aeruginosa": "p. aeruginosa",
    "abaumannii": "a. baumannii"
})

print("Unique pathogens found:")
print(df["Pathogen"].unique())


# ---------------------------------------------------
# 3. KEEP ONLY TARGET PATHOGENS
# ---------------------------------------------------

TARGET_PATHOGENS = [
    "e. coli",
    "k. pneumoniae",
    "s. aureus",
    "a. baumannii",
    "p. aeruginosa"
]

df = df[df["Pathogen"].isin(TARGET_PATHOGENS)]

print("After pathogen filtering:", df.shape)


# ---------------------------------------------------
# 4. USE NUMERIC RESISTANCE COLUMN
# ---------------------------------------------------

df["No_of_ABX_Classes_Resistant"] = pd.to_numeric(
    df["RESISTANT"],
    errors="coerce"
).fillna(0)

print("Resistance summary:")
print(df["No_of_ABX_Classes_Resistant"].describe())

# keep isolates with resistance ≥2
df = df[df["No_of_ABX_Classes_Resistant"] >= 2]

print("After resistance filtering:", df.shape)



# ---------------------------------------------------
# 5. REMOVE DUPLICATE ISOLATES
# ---------------------------------------------------

df = (
    df.sort_values("No_of_ABX_Classes_Resistant", ascending=False)
      .drop_duplicates(subset=["Isolate code","Pathogen"], keep="first")
)

print("After duplicate removal:", df.shape)


# ---------------------------------------------------
# 6. APPLY PI PATHOGEN QUOTA
# ---------------------------------------------------

QUOTA = {
    "e. coli":4,
    "k. pneumoniae":3,
    "s. aureus":3,
    "a. baumannii":2,
    "p. aeruginosa":2
}

selected = []

for pathogen, n in QUOTA.items():

    subset = df[df["Pathogen"] == pathogen]

    subset = subset.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    selected.append(subset.head(n))

result = pd.concat(selected)


# ---------------------------------------------------
# 7. FILL REMAINING SLOTS
# ---------------------------------------------------

remaining = 14 - len(result)

if remaining > 0:

    leftover = df[~df["Isolate code"].isin(result["Isolate code"])]

    leftover = leftover.sort_values(
        "No_of_ABX_Classes_Resistant",
        ascending=False
    )

    result = pd.concat([result, leftover.head(remaining)])


# ---------------------------------------------------
# 8. FINAL OUTPUT
# ---------------------------------------------------

sample_type_col = [c for c in df.columns if "sample type" in c.lower()][0]

result = result[[
    "Sample Code",
    "Isolate code",
    "Farm Name",
    sample_type_col,
    "Pathogen",
    "No_of_ABX_Classes_Resistant"
]]

result = result.sort_values(
    "No_of_ABX_Classes_Resistant",
    ascending=False
)

result.to_excel(OUTPUT_FILE, index=False)

print("\nEnvironmental selection complete")
print("Final dataset size:", result.shape)


Shape after loading: (313, 57)
Unique pathogens found:
['nan' 'e. coli' 's. aureus' 'k. pneumoniae' 's pneumoniae' 'a. baumannii'
 'p. aeruginosa' 'alwfii' 'calbicans' 's epidermidis']
After pathogen filtering: (253, 57)
Resistance summary:
count    253.000000
mean       0.300395
std        0.966030
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        6.000000
Name: No_of_ABX_Classes_Resistant, dtype: float64
After resistance filtering: (15, 58)
After duplicate removal: (15, 58)

Environmental selection complete
Final dataset size: (14, 6)
